---
title: Check what a model quotes
description: Number the comments of a Reddit thread export the way Comment Scraper's Markdown export does, ask an LLM for word-for-word quotes with citations, and check every quote against the comment it cites in Python.
---

Ask an LLM to pull quotes from a thread and you get something that looks right. Whether it is right is a separate question, and when every comment has a number, code can answer much of it. This chapter numbers the comments, sends them to a model with a prompt that asks for exact quotes, and checks each quote against the comment it cites.

In [ ]:
import json
import re

import pandas as pd

from threadtools import check_tree, load_export

pd.set_option("display.width", 100)
pd.set_option("display.max_colwidth", 70)

records, meta = load_export("data/sample_thread.json")
post, comments = check_tree(records)

## Number the comments

The model needs a short, stable handle for each comment. Comment Scraper's Markdown export numbers comments `[c1]`, `[c2]` and so on in file order, counting deleted ones too, so a number never shifts when you leave comments out. The [LangChain loader](https://comment-scraper.com/langchain) and the MCP server in the [Reddit-to-Claude guide](https://comment-scraper.com/reddit-to-claude) use the same numbers, so `[c12]` means the same comment wherever you read it.

```{literalinclude} threadtools.py
:start-at: def number_comments
:end-before: UNITS =
```

`thread_markdown` writes the whole thread in the export's Markdown layout: a header with the three numbers, the post, then each comment with its number, the number of the comment it replies to, its score and age, and whether the OP wrote it. I compared its output with the extension's own exporter on both sample threads, and the files are identical.

:::{dropdown} The code of thread_markdown
```{literalinclude} threadtools.py
:start-at: UNITS =
:end-before: # The extension's built-in
```
:::

In [ ]:
from threadtools import number_comments, thread_markdown

numbers = number_comments(records)
markdown = thread_markdown(records, meta.iloc[0])
print("\n".join(markdown.splitlines()[:22]))

## Ask for quotes

The prompt is the extension's built-in "Quotes for ad copy" prompt, word for word. Its Copy with prompt button puts it above the thread like this. It asks for 10 to 20 phrases, copied word for word, each with its citation:

In [ ]:
from threadtools import QUOTES_PROMPT, with_prompt

print(QUOTES_PROMPT)

In [ ]:
prompt = with_prompt(QUOTES_PROMPT, markdown)
# The saved answer below came from this exact prompt
assert prompt == open("data/llm/prompt.md", encoding="utf-8").read()
print(f"{len(prompt):,} characters, the same as data/llm/prompt.md")

I sent this prompt once to a small, inexpensive model, the kind you'd run over many threads, with `scripts/ask_model.py` (shown on [Run it yourself](run-it-yourself.md)), and saved the answer exactly as it came back. The book reads that file and never calls a model, so every build checks the same answer.

In [ ]:
json.load(open("data/llm/answer.json"))

:::{dropdown} The model's answer, as saved in data/llm/answer.md
```{literalinclude} data/llm/answer.md
:language: markdown
```
:::

It reads well: three groups of quoted phrases, a citation after each and some notes at the end. The prompt asked for 10 to 20 phrases and got 46, which is the first thing it didn't follow. Asked again, the same model would word things differently, so treat what follows as one example of the kind of mistakes to look for.

## Check every quote

For each quoted phrase, `check_quotes` looks up the comment it cites and asks four questions. Does that comment exist? Does it still have text? Is the phrase in it exactly as written, as whole words? Is it there once you ignore case, curly quotes, spacing and a final full stop? If not, it looks for the phrase in every other comment, to catch a quote filed under the wrong number. The whole-word rule matters: without it, a quote of "$12." would pass against a comment that says "$12.49".

```{literalinclude} threadtools.py
:start-at: CITATION =
:end-before: def quotes_not_checked
```

In [ ]:
from threadtools import check_quotes

answer = open("data/llm/answer.md", encoding="utf-8").read()
checked = check_quotes(answer, comments, numbers)

A quote the parser can't pair with citations would slip past every check, so first list the lines where that happened:

```{literalinclude} threadtools.py
:start-at: def quotes_not_checked
:end-before: def uncited_lines
```

In [ ]:
from threadtools import quotes_not_checked

quotes_not_checked(answer)

One line. The quote at its start was checked, but the note after it holds a second, uncited quote: "credits don't make dinner on a Wednesday", in lowercase. That one is the exact wording of `c28`. The model had the right text in hand, capitalised it in the quote it cited, and said in the same note that the quote was exact. Here is how the 46 cited quotes came out:

In [ ]:
checked.verdict.value_counts()

In [ ]:
assert len(checked) == 46
assert len(quotes_not_checked(answer)) == 1
assert checked.verdict.value_counts().to_dict() == {
    "exact": 40,
    "same words, different case or punctuation": 4,
    "not in this comment": 2,
}

Forty of the 46 quotes are exact, and none cites a comment that doesn't exist or a deleted one. Here are the other six, each next to the comment it cites:

In [ ]:
import textwrap

body_of = {f"c{numbers[cid]}": body
           for cid, body in zip(comments.comment_id, comments.body)}
for row in checked[checked.verdict != "exact"].itertuples():
    comment = " / ".join(line for line in body_of[row.cited].split("\n") if line)
    print(f"[{row.cited}] {row.verdict}")
    print(f"  quote:   {row.quote}")
    print(textwrap.fill(comment, 84, initial_indent="  comment: ",
                        subsequent_indent=" " * 11), "\n")

Four of the six differ only in case. The model capitalised the first word of a phrase it took from the middle of a sentence, and for `c49` it also dropped a leading "Yeah". That's fine for most uses. The other two fail. For `c64` the model joined two items of a bulleted list with a slash, and for `c74` it glued two sentences together with an ellipsis, dropping "Two kids, both parents working" from between them. The prompt said "Copy each phrase word for word" and "Do not rewrite or polish the quotes". Two quotes out of 46 aren't copied, and nothing in the answer tells you which two.

## Check what the answer says about itself

The answer ends with notes on how it was made. They carry no citations, so a separate check lists them:

In [ ]:
from threadtools import uncited_lines

for line in uncited_lines(answer):
    print(line, "\n")

Some of these can be checked against the data. The second note says the deleted and removed comments are c20, c40 and c81:

In [ ]:
gone = comments[comments.comment_state != "normal"]
[f"c{numbers[comment_id]}" for comment_id in gone.comment_id]

That's right. The same note also says that several of the quoted phrases are the OP's:

In [ ]:
cited = set(checked.cited)
quoted = comments[comments.comment_id.map(lambda cid: f"c{numbers[cid]}" in cited)]
print(f"{len(quoted)} comments quoted, {int(quoted.is_op.sum())} of them by the OP")

In [ ]:
assert len(quoted) == 39 and quoted.is_op.sum() == 0

None of them is. The OP's five comments are all marked `OP` in the Markdown the model read, and it quoted none of them. The last note says c28 "is quoted as written". The check above found it differs in case, a small error, but it shows the notes are as much a guess as the rest.

## What the check can't tell you

A quote can be exact and still misleading. The check proves the words are in the comment. It can't tell whether the comment says what the answer's grouping implies. "Beginners have empty pantries" is filed under "comparisons with alternatives". Read in place, it's the last reply in the argument from the second chapter about whether a kit costs more than groceries:

In [ ]:
from threadtools import show_in_context

number_to_id = {number: comment_id for comment_id, number in numbers.items()}
show_in_context(number_to_id[11], comments, width=72)

It also can't see what the model left out. It quoted 39 of the 92 comments that have text. The median score of the quoted comments is well above the thread's:

In [ ]:
with_text = comments[comments.comment_state == "normal"]
pd.Series({"quoted comments": quoted.score.median(),
           "all comments with text": with_text.score.median()}, name="median score")

The prompt never mentioned scores, but they're in the Markdown, and the answer leans towards the comments people upvoted. If the quieter comments matter to you, ask for them or read them yourself.

So here is the split. Code can check that every citation exists, that every quote is really in the comment it cites, and some of the answer's claims about itself. A person still has to check whether each quote means what the answer says it means. Comment Scraper's AI brief removes the first problem by design: the model returns only comment IDs, and the extension takes the quote text from your export, so a brief can't contain a made-up quote. The [help pages](https://comment-scraper.com/docs#ai-briefs-and-credits) explain how briefs work.